## Feature Engineering for Survival Analysis

### Objective
This notebook prepares the METABRIC clinical dataset for survival modeling.

Main goals:
- define modeling features
- clean numeric and categorical variables
- encode categorical variables
- construct a Cox PH-ready dataset

---

## 목적
이 노트북의 목적은 METABRIC 임상 데이터를 survival modeling에 적합한 형태로 변환하는 것이다.

핵심 목표:
- 모델링에 사용할 feature 확정
- 수치형 / 범주형 변수 정리
- 범주형 변수 인코딩
- Cox Proportional Hazards 모델 입력용 데이터셋 생성

In [26]:
# 1. import
import numpy as np 
import pandas as pd 

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [27]:
# 2. Load data 
file_path = "../data/raw/brca_metabric_clinical_data.tsv"
df = pd.read_csv(file_path, sep="\t")

print("Original shape:", df.shape)
df.head()

Original shape: (2509, 39)


,Study ID,Patient ID,Sample ID,Age at Diagnosis,Type of Breast Surgery,Cancer Type,Cancer Type Detailed,Cellularity,Chemotherapy,Pam50 + Claudin-low subtype,Cohort,ER status measured by IHC,ER Status,Neoplasm Histologic Grade,HER2 status measured by SNP6,HER2 Status,Tumor Other Histologic Subtype,Hormone Therapy,Inferred Menopausal State,Integrative Cluster,Primary Tumor Laterality,Lymph nodes examined positive,Mutation Count,Nottingham prognostic index,Oncotree Code,Overall Survival (Months),Overall Survival Status,PR Status,Radio Therapy,Relapse Free Status (Months),Relapse Free Status,Number of Samples Per Patient,Sample Type,Sex,3-Gene classifier subtype,TMB (nonsynonymous),Tumor Size,Tumor Stage,Patient's Vital Status
0,brca_metabric,MB-0000,MB-0000,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,NO,claudin-low,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,YES,Post,4ER+,Right,10.0,NaN,6.044,IDC,140.500000,0:LIVING,Negative,YES,140.500000,0:Not Recurred,1,Primary,Female,ER-/HER2-,0.000000,22.0,2.0,Living
1,brca_metabric,MB-0002,MB-0002,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,NO,LumA,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,YES,Pre,4ER+,Right,0.0,2.0,4.020,IDC,84.633333,0:LIVING,Positive,YES,84.633333,0:Not Recurred,1,Primary,Female,ER+/HER2- High Prolif,2.615035,10.0,1.0,Living
2,brca_metabric,MB-0005,MB-0005,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,YES,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Ductal/NST,YES,Pre,3,Right,1.0,2.0,4.030,IDC,163.700000,1:DECEASED,Positive,NO,153.300000,1:Recurred,1,Primary,Female,NaN,2.615035,15.0,2.0,Died of Disease
3,brca_metabric,MB-0006,MB-0006,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,YES,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Mixed,YES,Pre,9,Right,3.0,1.0,4.050,MDLC,164.933333,0:LIVING,Positive,YES,164.933333,0:Not Recurred,1,Primary,Female,NaN,1.307518,25.0,2.0,Living
4,brca_metabric,MB-0008,MB-0008,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,YES,LumB,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Mixed,YES,Post,9,Right,8.0,2.0,6.080,MDLC,41.366667,1:DECEASED,Positive,YES,18.800000,1:Recurred,1,Primary,Female,ER+/HER2- High Prolif,2.615035,40.0,2.0,Died of Disease


In [28]:
# 3. Define target variables
df["event"] = df["Patient's Vital Status"].apply(
    lambda x: 1 if x == "Died of Disease" else 0
)

df["time"] = pd.to_numeric(df["Overall Survival (Months)"], errors="coerce")

df[["Patient's Vital Status", "Overall Survival (Months)", "event", "time"]].head()

,Patient's Vital Status,Overall Survival (Months),event,time
0,Living,140.500000,0,140.500000
1,Living,84.633333,0,84.633333
2,Died of Disease,163.700000,1,163.700000
3,Living,164.933333,0,164.933333
4,Died of Disease,41.366667,1,41.366667


## Initial Feature Set

We start with a clinically interpretable feature set.

Selected variables:
- Age at Diagnosis
- Neoplasm Histologic Grade
- Tumor Stage
- ER Status
- HER2 Status
- Lymph nodes examined positive
- Tumor Size
- time
- event

---

# 초기 변수 선택 기준
초기 변수셋은 임상적으로 해석 가능하고 survival outcome과 관련성이 높은 변수들로 구성한다.

선택 변수:
- 진단 시 나이
- 조직학적 grade
- 종양 stage
- ER 상태
- HER2 상태
- 양성 림프절 수
- 종양 크기
- time
- event

In [29]:
# 4. Build modeling dataframe
selected_features = [
    "Age at Diagnosis",
    "Neoplasm Histologic Grade",
    "Tumor Stage",
    "ER Status",
    "HER2 Status",
    "Lymph nodes examined positive",
    "Tumor Size",
    "time",
    "event",
]

df_model = df[selected_features].copy()

print("Before cleaning:", df_model.shape)
df_model.head()

Before cleaning: (2509, 9)


,Age at Diagnosis,Neoplasm Histologic Grade,Tumor Stage,ER Status,HER2 Status,Lymph nodes examined positive,Tumor Size,time,event
0,75.65,3.0,2.0,Positive,Negative,10.0,22.0,140.500000,0
1,43.19,3.0,1.0,Positive,Negative,0.0,10.0,84.633333,0
2,48.87,2.0,2.0,Positive,Negative,1.0,15.0,163.700000,1
3,47.68,2.0,2.0,Positive,Negative,3.0,25.0,164.933333,0
4,76.97,3.0,2.0,Positive,Negative,8.0,40.0,41.366667,1


In [30]:
# 5. Check missing values
missing_summary = df_model.isnull().sum().sort_values(ascending=False)
missing_summary

Tumor Stage                      721
HER2 Status                      529
time                             528
Lymph nodes examined positive    266
Tumor Size                       149
Neoplasm Histologic Grade        121
ER Status                         40
Age at Diagnosis                  11
event                              0
dtype: int64

In [31]:
# 6. Drop missing rows for core modeling set
df_model = df_model.dropna().copy()

print("After dropna:", df_model.shape)
df_model.head()

After dropna: (1354, 9)


,Age at Diagnosis,Neoplasm Histologic Grade,Tumor Stage,ER Status,HER2 Status,Lymph nodes examined positive,Tumor Size,time,event
0,75.65,3.0,2.0,Positive,Negative,10.0,22.0,140.500000,0
1,43.19,3.0,1.0,Positive,Negative,0.0,10.0,84.633333,0
2,48.87,2.0,2.0,Positive,Negative,1.0,15.0,163.700000,1
3,47.68,2.0,2.0,Positive,Negative,3.0,25.0,164.933333,0
4,76.97,3.0,2.0,Positive,Negative,8.0,40.0,41.366667,1


In [32]:
# 7. Inspect unique values in categorical columns
categorical_cols = [
    "Neoplasm Histologic Grade",
    "Tumor Stage",
    "ER Status",
    "HER2 Status",
]

for col in categorical_cols:
    print(f"\n[{col}]")
    print(sorted(df_model[col].astype(str).unique()))


[Neoplasm Histologic Grade]
['1.0', '2.0', '3.0']

[Tumor Stage]
['0.0', '1.0', '2.0', '3.0', '4.0']

[ER Status]
['Negative', 'Positive']

[HER2 Status]
['Negative', 'Positive']


In [33]:
# 8. Clean categorical variables
# Tumor Stage or Grade = 문자열/숫자 혼합 가능성이 있어서 문자열로 통일함

df_model["Neoplasm Histologic Grade"] = df_model["Neoplasm Histologic Grade"].astype(str).str.strip()
df_model["Tumor Stage"] = df_model["Tumor Stage"].astype(str).str.strip()
df_model["ER Status"] = df_model["ER Status"].astype(str).str.strip()
df_model["HER2 Status"] = df_model["HER2 Status"].astype(str).str.strip()

In [34]:
# Remove rare and clinically ambiguous Tumor Stage = 0.0
df_model = df_model[df_model["Tumor Stage"] != "0.0"].copy()

print("Shape after removing Tumor Stage == 0.0:", df_model.shape)
print("\nTumor Stage distribution:")
print(df_model["Tumor Stage"].value_counts().sort_index())

Shape after removing Tumor Stage == 0.0: (1353, 9)

Tumor Stage distribution:
Tumor Stage
1.0    457
2.0    776
3.0    111
4.0      9
Name: count, dtype: int64


## Handling Rare Tumor Stage Category

A single record with `Tumor Stage = 0.0` was identified.

Because this category is clinically ambiguous and contains only one sample,  
it was removed before one-hot encoding to avoid using it as an unintended reference category.

---

## 희귀 Tumor Stage 범주 처리

`Tumor Stage = 0.0` 인 샘플이 1건 확인되었다.

이 범주는 임상적으로 해석이 모호하고 표본 수도 1건뿐이므로,  
one-hot encoding 이전에 제거하여 의도하지 않은 기준범주(reference category)로 사용되는 것을 방지하였다.

In [35]:
# 9. Ensure numeric columns are numeric
numeric_cols = [
    "Age at Diagnosis",
    "Lymph nodes examined positive",
    "Tumor Size",
    "time",
    "event",
]

for col in numeric_cols:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

df_model[numeric_cols].dtypes

Age at Diagnosis                 float64
Lymph nodes examined positive    float64
Tumor Size                       float64
time                             float64
event                              int64
dtype: object

In [36]:
# 10. Final NA check after type conversion
df_model = df_model.dropna().copy()

print("Shape after numeric conversion + dropna:", df_model.shape)
df_model.info()

Shape after numeric conversion + dropna: (1353, 9)
<class 'pandas.core.frame.DataFrame'>
Index: 1353 entries, 0 to 1743
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age at Diagnosis               1353 non-null   float64
 1   Neoplasm Histologic Grade      1353 non-null   object 
 2   Tumor Stage                    1353 non-null   object 
 3   ER Status                      1353 non-null   object 
 4   HER2 Status                    1353 non-null   object 
 5   Lymph nodes examined positive  1353 non-null   float64
 6   Tumor Size                     1353 non-null   float64
 7   time                           1353 non-null   float64
 8   event                          1353 non-null   int64  
dtypes: float64(4), int64(1), object(4)
memory usage: 105.7+ KB


## Feature Engineering Strategy

For Cox Proportional Hazards modeling:

- Numerical variables will be kept in continuous form for interpretability.
- Categorical variables will be one-hot encoded.
- `time` and `event` will remain unchanged as survival targets.
- A reference category will be dropped to avoid redundancy.

---

# Feature Engineering 전략

Cox PH 모델링을 위해 다음 원칙을 적용한다.

- 수치형 변수는 해석 가능성을 위해 연속형으로 유지
- 범주형 변수는 one-hot encoding 적용
- `time`, `event`는 survival target으로 그대로 유지
- 기준 범주 하나는 제거하여 중복을 방지

In [37]:
# 11. Separate feature types
categorical_cols = [
    "Neoplasm Histologic Grade",
    "Tumor Stage",
    "ER Status",
    "HER2 Status",
]

numeric_feature_cols = [
    "Age at Diagnosis",
    "Lymph nodes examined positive",
    "Tumor Size",
]

In [38]:
# 12. One-hot encoding
df_encoded = pd.get_dummies(
    df_model,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print("Encoded shape:", df_encoded.shape)
df_encoded.head()

Encoded shape: (1353, 12)


,Age at Diagnosis,Lymph nodes examined positive,Tumor Size,time,event,Neoplasm Histologic Grade_2.0,Neoplasm Histologic Grade_3.0,Tumor Stage_2.0,Tumor Stage_3.0,Tumor Stage_4.0,ER Status_Positive,HER2 Status_Positive
0,75.65,10.0,22.0,140.500000,0,0,1,1,0,0,1,0
1,43.19,0.0,10.0,84.633333,0,0,1,0,0,0,1,0
2,48.87,1.0,15.0,163.700000,1,1,0,1,0,0,1,0
3,47.68,3.0,25.0,164.933333,0,1,0,1,0,0,1,0
4,76.97,8.0,40.0,41.366667,1,0,1,1,0,0,1,0


In [39]:
# 13. Check encoded columns
df_encoded.columns.tolist()

['Age at Diagnosis',
 'Lymph nodes examined positive',
 'Tumor Size',
 'time',
 'event',
 'Neoplasm Histologic Grade_2.0',
 'Neoplasm Histologic Grade_3.0',
 'Tumor Stage_2.0',
 'Tumor Stage_3.0',
 'Tumor Stage_4.0',
 'ER Status_Positive',
 'HER2 Status_Positive']

In [40]:
# 14. Validate final modeling dataset
print("Any missing values:", df_encoded.isnull().sum().sum())
print("\nData types:")
print(df_encoded.dtypes)

Any missing values: 0

Data types:
Age at Diagnosis                 float64
Lymph nodes examined positive    float64
Tumor Size                       float64
time                             float64
event                              int64
Neoplasm Histologic Grade_2.0      int64
Neoplasm Histologic Grade_3.0      int64
Tumor Stage_2.0                    int64
Tumor Stage_3.0                    int64
Tumor Stage_4.0                    int64
ER Status_Positive                 int64
HER2 Status_Positive               int64
dtype: object


In [41]:
# 15. Reorder colums for modeling clarity
# target_cols = 뒤로 보내는 것보다, survival modeling에서는 time, event를 뒤에 두면 보기 좋다.

feature_cols = [col for col in df_encoded.columns if col not in ["time", "event"]]
df_final = df_encoded[feature_cols + ["time", "event"]].copy()

print("Final Cox-ready dataset shape:", df_final.shape)
df_final.head()

Final Cox-ready dataset shape: (1353, 12)


,Age at Diagnosis,Lymph nodes examined positive,Tumor Size,Neoplasm Histologic Grade_2.0,Neoplasm Histologic Grade_3.0,Tumor Stage_2.0,Tumor Stage_3.0,Tumor Stage_4.0,ER Status_Positive,HER2 Status_Positive,time,event
0,75.65,10.0,22.0,0,1,1,0,0,1,0,140.500000,0
1,43.19,0.0,10.0,0,1,0,0,0,1,0,84.633333,0
2,48.87,1.0,15.0,1,0,1,0,0,1,0,163.700000,1
3,47.68,3.0,25.0,1,0,1,0,0,1,0,164.933333,0
4,76.97,8.0,40.0,0,1,1,0,0,1,0,41.366667,1


In [42]:
# 13. Save processed dataset (Code)
# 경로는 projects/survival-analysis-breast-cancer/data/processed/ 기준.

output_path = "../data/processed/metabric_clinical_featurized.csv"
df_final.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ../data/processed/metabric_clinical_featurized.csv


In [43]:
# 14. Feature summary table
feature_summary = pd.DataFrame({
    "column": df_final.columns,
    "dtype": df_final.dtypes.astype(str).values
})

feature_summary

,column,dtype
0,Age at Diagnosis,float64
1,Lymph nodes examined positive,float64
2,Tumor Size,float64
3,Neoplasm Histologic Grade_2.0,int64
4,Neoplasm Histologic Grade_3.0,int64
5,Tumor Stage_2.0,int64
6,Tumor Stage_3.0,int64
7,Tumor Stage_4.0,int64
8,ER Status_Positive,int64
9,HER2 Status_Positive,int64


## Feature Engineering Summary

The final dataset is now ready for Cox Proportional Hazards modeling.

Key processing steps:
- survival target variables (`time`, `event`) were preserved
- missing values were removed from the core feature set
- categorical variables were one-hot encoded
- numerical variables were retained in continuous form
- the final dataset was exported for downstream modeling

---

## Feature Engineering 요약

최종 데이터셋은 이제 Cox Proportional Hazards 모델링에 바로 사용할 수 있는 상태이다.

주요 처리 내용:
- survival target 변수(`time`, `event`) 유지
- 핵심 변수셋 기준 결측치 제거
- 범주형 변수 one-hot encoding 적용
- 수치형 변수는 연속형 유지
- 최종 모델링용 데이터셋 저장 완료

---

## Why This Step Matters

This stage is important because survival models are sensitive to:
- invalid data types
- missing values
- inconsistent categorical labels
- redundant encoded variables

By standardizing the feature space here, we improve model stability, interpretability, and reproducibility.

---

## 왜 이 단계가 중요한가

survival model은 다음 문제에 민감하다.
- 잘못된 데이터 타입
- 결측치
- 일관되지 않은 범주형 라벨
- 중복 인코딩 변수

따라서 이 단계에서 feature space를 정리하면
모델의 안정성, 해석 가능성, 재현 가능성을 높일 수 있다.

In [44]:
# 14 Optional sanity checks

print(df_final.shape)
print(df_final[["time", "event"]].describe())

(1353, 12)
              time        event
count  1353.000000  1353.000000
mean    127.110273     0.338507
std      78.047434     0.473377
min       0.100000     0.000000
25%      61.466667     0.000000
50%     116.633333     0.000000
75%     187.933333     1.000000
max     351.000000     1.000000
